In [ ]:

import os, sys, glob, shutil, time, json, gc
import numpy as np, pandas as pd
t0=time.perf_counter()
def log(m): print(f"[{time.perf_counter()-t0:7.0f}с] {m}", flush=True)
base=os.path.dirname(glob.glob("/kaggle/input/**/items_human.parquet", recursive=True)[0])
os.makedirs("/kaggle/working/src",exist_ok=True)
for p in glob.glob(base+"/*.py"): shutil.copy(p,"/kaggle/working/src/")
open("/kaggle/working/src/__init__.py","a").close()
os.makedirs("/kaggle/working/models",exist_ok=True)
shutil.copy(base+"/anti_words.json","/kaggle/working/models/anti_words.json")
os.chdir("/kaggle/working"); sys.path.insert(0,"/kaggle/working")
from src.attr_features import parse, compare, FEATURE_NAMES
from src.name_features import parse_name, compare_names, build_idf, NAME_FEATURE_NAMES
from src.string_features import compare_strings, STRING_FEATURE_NAMES
from src.neighbour_features import build as nb_build, compare as nb_compare, NEIGHBOUR_FEATURE_NAMES
from src.brand_features import colours, canonical, compare_brands, compare_colours, BRAND_FEATURE_NAMES
from src.dim_features import dimensions, compare_dimensions, compare_translit, DIM_FEATURE_NAMES
from src.hybrid import product_disjoint_pair_masks
from src.metrics import macro_pr_auc
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import average_precision_score

ALL=FEATURE_NAMES+NAME_FEATURE_NAMES+STRING_FEATURE_NAMES+NEIGHBOUR_FEATURE_NAMES+BRAND_FEATURE_NAMES+DIM_FEATURE_NAMES
log(f"признаков: {len(ALL)}")

def featurize(items, frames, tag):
    """Признаки для нескольких наборов пар по одному пулу товаров, по категориям."""
    ID=items["id"].to_numpy(); NAME=items["name"].astype(str).tolist(); ATTR=items["attributes"].tolist()
    CAT=items["category"].astype(str).to_numpy()
    cards={int(i):parse(n,a,name=n) for i,n,a in zip(ID,NAME,ATTR)}
    names={int(i):parse_name(n) for i,n in zip(ID,NAME)}
    idf,avg=build_idf(list(names.values()))
    cols={int(i):colours(n+" "+str(a)) for i,n,a in zip(ID,NAME,ATTR)}
    brand={int(i):frozenset(x for x in (canonical(v) for v in c.slots.get("brand",())) if x) for i,c in cards.items()}
    dims={int(i):dimensions(n+" "+str(a)) for i,n,a in zip(ID,NAME,ATTR)}
    cat_of=dict(zip(ID.tolist(),CAT.tolist())); POS={int(x):r for r,x in enumerate(ID)}
    log(f"{tag}: карточки разобраны ({len(cards):,})")
    outs=[np.zeros((len(f),len(ALL)),dtype=np.float32) for f in frames]
    for cat in sorted(set(CAT)):
        prof=nb_build(items[["id","name","category"]], categories=[cat])
        g=items[items["category"]==cat]
        M=TfidfVectorizer(min_df=1,sublinear_tf=True).fit_transform(g["name"].astype(str).tolist())
        pos={int(x):r for r,x in enumerate(g["id"].to_numpy())}
        for fi,f in enumerate(frames):
            rows=np.flatnonzero(f["id1"].map(cat_of).astype(str).to_numpy()==cat)
            for r in rows:
                a,b=int(f["id1"].iat[r]),int(f["id2"].iat[r])
                s=float((M[pos[a]]@M[pos[b]].T).toarray()[0,0]) if (a in pos and b in pos) else 0.0
                d=compare(cards[a],cards[b]); d.update(compare_names(names[a],names[b],idf,avg))
                d.update(compare_strings(NAME[POS[a]],NAME[POS[b]]))
                d.update(nb_compare(a,b,s,prof))
                d.update(compare_brands(brand[a],brand[b],{})); d.update(compare_colours(cols[a],cols[b]))
                d.update(compare_dimensions(dims[a],dims[b])); d.update(compare_translit(brand[a],brand[b]))
                outs[fi][r]=[d[k] for k in ALL]
        del prof, M; gc.collect()
        log(f"{tag}: {cat} готова")
    return outs

hitems=pd.read_parquet(base+"/items_human.parquet")
hm=pd.read_parquet(base+"/matches.parquet",columns=["id1","id2","target"])
ev1=pd.read_parquet(base+"/eval_pairs.parquet"); ev2=pd.read_parquet(base+"/eval_pairs_mixed.parquet")
Xh,E1,E2=featurize(hitems,[hm,ev1,ev2],"ручные")
for n,a in (("features_human",Xh),("features_eval_lex",E1),("features_eval_mixed",E2)):
    np.save(f"/kaggle/working/{n}.npy",a)
json.dump(list(ALL),open("/kaggle/working/feature_names.json","w"),ensure_ascii=False)
log("ручные признаки сохранены")

lp=pd.read_parquet(base+"/llm_pairs_sel.parquet"); li=pd.read_parquet(base+"/llm_items_sel.parquet")
Xl,=featurize(li,[lp],"LLM"); np.save("/kaggle/working/features_llm.npy",Xl)
yl=lp["label"].to_numpy(np.int8); del li; gc.collect()
log("LLM признаки сохранены")

y=hm["target"].to_numpy(np.int8)
cat_of=dict(zip(hitems["id"],hitems["category"].astype(str)))
cp=hm["id1"].map(cat_of).astype(str).to_numpy()
tm,vm=product_disjoint_pair_masks(hm["id1"].to_numpy(),hm["id2"].to_numpy(),0,3)
tr,va=np.flatnonzero(tm),np.flatnonzero(vm)
S=np.load(base+"/human_features_v13.npy")
lab=sorted(set(cp)|set(ev1["category"].astype(str))|set(ev2["category"].astype(str)))
code={k:i for i,k in enumerate(lab)}
def withcat(X,cats): return np.hstack([X,np.asarray([code.get(k,-1) for k in cats],dtype=np.float32).reshape(-1,1)])
c1=ev1["category"].astype(str).to_numpy(); c2=ev2["category"].astype(str).to_numpy()
y1=ev1["target"].to_numpy(np.int8); y2=ev2["target"].to_numpy(np.int8)
def macro(p,c,yy): return float(np.mean([average_precision_score(yy[c==k],p[c==k])
    for k in np.unique(c) if len(np.unique(yy[c==k]))>1]))

Xfull=withcat(np.hstack([S,Xh]),cp)
E1f=withcat(np.hstack([np.zeros((len(ev1),S.shape[1]),np.float32),E1]),c1)
E2f=withcat(np.hstack([np.zeros((len(ev2),S.shape[1]),np.float32),E2]),c2)
ci=[Xfull.shape[1]-1]
log("обучение")
for tag,Xt,yt in (("ручные",Xfull[tr],y[tr]),
                  ("LLM",withcat(np.hstack([np.zeros((len(Xl),S.shape[1]),np.float32),Xl]),
                                 lp["id1"].map(cat_of).fillna("?").astype(str).to_numpy()),yl)):
    clf=HistGradientBoostingClassifier(max_iter=400,learning_rate=0.08,random_state=0,
                                       categorical_features=ci).fit(Xt,yt)
    p=clf.predict_proba(Xfull[va])[:,1]
    p1=clf.predict_proba(E1f)[:,1]; p2=clf.predict_proba(E2f)[:,1]
    np.save(f"/kaggle/working/pred_{tag}_lex.npy",p1); np.save(f"/kaggle/working/pred_{tag}_mixed.npy",p2)
    log(f"{tag:<8} holdout {macro_pr_auc(y[va],p,cp[va])[0]:.6f}  лексич {macro(p1,c1,y1):.6f}  смеш {macro(p2,c2,y2):.6f}")
log("готово")
